
# 00 — Data Acquisition

This is the first notebook in the final Economic Resilience project pipeline.

## Final decisions reflected in this notebook

- Output file names also use **`2002_2025`** so the file names match the actual temporal coverage.
- No country sample is selected here.
- No imputation is done here.
- No clipping is done here.
- No direction alignment is done here.
- No POSet variables are created here.
- Raw/source-level values are preserved as much as possible.
- WGI/governance is downloaded as contextual/sensitivity material only.
- WGI/governance is **not** part of the final POSet ordering variables.

## Final POSet ordering concepts supported downstream

The final POSet ordering will use five structural variables:

1. public debt / GDP → debt capacity
2. unemployment rate → employment strength
3. R&D / GDP → R&D intensity
4. tertiary attainment age 25–64 → human capital
5. energy import dependency → energy security

## Raw acquisition outputs

This notebook downloads:

- OECD R&D/GDP
- OECD inflation CPI
- OECD unemployment rate
- OECD adult tertiary attainment, age 25–64
- OECD productivity / GDP per hour worked
- OECD GDP growth
- Eurostat public debt
- World Bank public debt
- World Bank raw energy import dependency
- World Bank WGI raw package

## Output folder

All raw files are saved to:

```text
Data/Raw/Raw_Files/
```

## Temporal coverage

The acquisition request is:

```python
START_YEAR = 2002
END_YEAR = 2025
```

Some sources naturally end before 2025. This is normal.



## Methodological guardrails

This notebook is deliberately only an acquisition notebook.

The following are **not** done here:

- final country selection;
- final variable selection;
- transformation into 0–100 scores;
- inversion of negative-direction indicators;
- POSet construction;
- recovery validation.

The WGI dataset is retained because it can still be useful in the report as institutional context or sensitivity analysis. It is **not** one of the final five POSet ordering variables.


In [1]:
# ------------------------------------------------------
# IMPORTS AND PATHS
# ------------------------------------------------------

from pathlib import Path
from datetime import datetime
from urllib.parse import quote
import requests
import pandas as pd
import numpy as np
import io
import json
import time
import re
import zipfile
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 150)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", "{:.4f}".format)

PROJECT_ROOT = Path.cwd()

RAW_FILES_DIR = PROJECT_ROOT / "Data" / "Raw" / "Raw_Files"
RAW_FILES_DIR.mkdir(parents=True, exist_ok=True)

DIAGNOSTICS_DIR = PROJECT_ROOT / "Data" / "Diagnostics" / "00A_Data_Acquisition"
DIAGNOSTICS_DIR.mkdir(parents=True, exist_ok=True)

AUDIT_DIR = PROJECT_ROOT / "Data" / "Audit"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_TIMESTAMP = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

START_YEAR = 2002
END_YEAR = 2025

OVERWRITE_RAW_FILES = True

# During development, keep this False so one broken source does not destroy the whole run.
# Once all sources are stable, you can set it to True.
STOP_ON_REQUIRED_FAILURE = False

print("Run ID:", RUN_ID)
print("Project root:", PROJECT_ROOT.resolve())
print("Raw files folder:", RAW_FILES_DIR.resolve())
print("Diagnostics folder:", DIAGNOSTICS_DIR.resolve())
print("Audit folder:", AUDIT_DIR.resolve())

Run ID: 20260624_173824
Project root: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24
Raw files folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Raw\Raw_Files
Diagnostics folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Diagnostics\00A_Data_Acquisition
Audit folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Audit


In [2]:
# ------------------------------------------------------
# API SETTINGS
# ------------------------------------------------------

# OECD REST
OECD_BASE_URL = "https://sdmx.oecd.org/public/rest/data"
OECD_VERSION = "all"

OECD_HEADERS = {
    "Accept": "application/vnd.sdmx.data+csv; charset=utf-8; version=2",
    "User-Agent": "EconomicResilienceProject/1.0 (Python/Requests)",
}

# DBnomics
DBNOMICS_BASE_URL = "https://api.db.nomics.world/v22"

DBNOMICS_HEADERS = {
    "Accept": "application/json",
    "User-Agent": "EconomicResilienceProject/1.0 (Python/Requests)",
}

# World Bank API
WORLD_BANK_BASE_URL = "https://api.worldbank.org/v2"

# Eurostat API
EUROSTAT_BASE_URL = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data"

REQUEST_TIMEOUT_SECONDS = 120
MAX_RETRIES = 4

print("API settings loaded.")

API settings loaded.


In [3]:
# ------------------------------------------------------
# ACQUISITION SAMPLE RULE
# ------------------------------------------------------
# We do NOT filter countries during acquisition.
#
# This notebook pulls every available country/geography for each selected
# indicator definition.
#
# Country inclusion/exclusion will be decided later in:
#
# 02_Raw_Files_Coverage_Diagnostics.ipynb
# 05_Master_Dataset_Build.ipynb
#
# Indicator-definition filters are allowed here.
# Country filters are not.

print("Acquisition rule: no country/sample filtering at this stage.")

Acquisition rule: no country/sample filtering at this stage.


In [4]:
# ------------------------------------------------------
# REQUEST HELPER WITH RETRY
# ------------------------------------------------------

def request_with_retry(
    url,
    params=None,
    headers=None,
    max_retries=MAX_RETRIES,
    timeout=REQUEST_TIMEOUT_SECONDS,
    sleep_base_seconds=10,
):
    retry_statuses = {403, 408, 429, 500, 502, 503, 504}
    last_response = None

    for attempt in range(1, max_retries + 1):
        try:
            response = requests.get(
                url,
                params=params,
                headers=headers,
                timeout=timeout,
            )

            last_response = response

            if response.status_code not in retry_statuses:
                return response

            wait = sleep_base_seconds * attempt
            print(
                f"  -> Temporary HTTP {response.status_code}. "
                f"Waiting {wait}s before retry {attempt}/{max_retries}..."
            )
            time.sleep(wait)

        except requests.RequestException as e:
            wait = sleep_base_seconds * attempt
            print(
                f"  -> Request error: {e}. "
                f"Waiting {wait}s before retry {attempt}/{max_retries}..."
            )
            time.sleep(wait)

    if last_response is not None:
        return last_response

    raise RuntimeError(f"Request failed repeatedly with no response: {url}")

In [5]:
# ------------------------------------------------------
# GENERAL HELPER FUNCTIONS
# ------------------------------------------------------

acquisition_log_rows = []
failed_download_rows = []


def safe_file_token(text):
    text = str(text)
    text = re.sub(r"[^A-Za-z0-9_]+", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")
    return text[:80]


def parse_annual_year(series, dataset_label):
    raw = series.astype(str).str.strip()

    invalid = raw[~raw.str.match(r"^\d{4}$", na=False)]

    if len(invalid) > 0:
        debug_path = DIAGNOSTICS_DIR / f"invalid_time_period__{safe_file_token(dataset_label)}.csv"
        pd.DataFrame({
            "invalid_time_period": sorted(invalid.unique().tolist())
        }).to_csv(debug_path, index=False)

        raise ValueError(
            f"{dataset_label}: TIME_PERIOD contains non-annual values. "
            f"Debug saved to {debug_path}"
        )

    return raw.astype(int)


def validate_tidy_country_year_value(df, dataset_name, stop_on_duplicates=True):
    required = {"Country", "Year", "Value"}
    missing = required - set(df.columns)

    if missing:
        raise ValueError(f"{dataset_name}: missing required columns {missing}")

    out = df.copy()

    out["Country"] = out["Country"].astype(str).str.strip().str.upper()
    out["Year"] = pd.to_numeric(out["Year"], errors="coerce")
    out["Value"] = pd.to_numeric(out["Value"], errors="coerce")

    out = out.dropna(subset=["Country", "Year", "Value"]).copy()
    out = out[out["Country"].astype(str).str.strip() != ""].copy()
    out["Year"] = out["Year"].astype(int)

    out = out[
        (out["Year"] >= START_YEAR)
        & (out["Year"] <= END_YEAR)
    ].copy()

    out = out.drop_duplicates().copy()

    duplicate_summary = (
        out.groupby(["Country", "Year"])
        .size()
        .reset_index(name="row_count")
        .query("row_count > 1")
        .sort_values(["row_count", "Country", "Year"], ascending=[False, True, True])
    )

    if len(duplicate_summary) > 0:
        duplicate_summary_path = DIAGNOSTICS_DIR / f"duplicate_country_year_summary__{safe_file_token(dataset_name)}.csv"
        duplicate_summary.to_csv(duplicate_summary_path, index=False)

        duplicate_keys = duplicate_summary[["Country", "Year"]]
        duplicate_detail = out.merge(duplicate_keys, on=["Country", "Year"], how="inner")

        duplicate_detail_path = DIAGNOSTICS_DIR / f"duplicate_country_year_detail__{safe_file_token(dataset_name)}.csv"
        duplicate_detail.to_csv(duplicate_detail_path, index=False)

        message = (
            f"{dataset_name}: duplicate Country-Year rows detected. "
            f"Summary saved to {duplicate_summary_path}"
        )

        if stop_on_duplicates:
            raise ValueError(message)
        else:
            print("WARNING:", message)

    return out.sort_values(["Country", "Year"]).reset_index(drop=True)


def write_tidy_csv(df, output_path, overwrite=True):
    output_path = Path(output_path)

    if output_path.exists() and overwrite:
        output_path.unlink()

    df.to_csv(output_path, index=False)
    return output_path


def add_acquisition_log(
    dataset_label,
    file_name,
    status,
    source_route,
    rows_raw=None,
    rows_saved=None,
    countries=None,
    min_year=None,
    max_year=None,
    output_file=None,
    note=None,
    url=None,
):
    acquisition_log_rows.append({
        "run_id": RUN_ID,
        "timestamp": RUN_TIMESTAMP,
        "dataset_label": dataset_label,
        "file_name": file_name,
        "status": status,
        "source_route": source_route,
        "rows_raw": rows_raw,
        "rows_saved": rows_saved,
        "countries": countries,
        "min_year": min_year,
        "max_year": max_year,
        "output_file": str(output_file) if output_file is not None else "",
        "note": note or "",
        "url": url or "",
    })


def add_failed_download(dataset_label, source_route, error, url=None):
    failed_download_rows.append({
        "run_id": RUN_ID,
        "timestamp": RUN_TIMESTAMP,
        "dataset_label": dataset_label,
        "source_route": source_route,
        "error": str(error),
        "url": url or "",
    })


def summarize_saved_csv(path, dataset_label):
    path = Path(path)

    if not path.exists():
        return {
            "dataset_label": dataset_label,
            "file_name": path.name,
            "exists": False,
            "rows": 0,
            "columns": 0,
            "countries": 0,
            "min_year": np.nan,
            "max_year": np.nan,
        }

    df = pd.read_csv(path)

    return {
        "dataset_label": dataset_label,
        "file_name": path.name,
        "exists": True,
        "rows": len(df),
        "columns": len(df.columns),
        "countries": df["Country"].nunique() if "Country" in df.columns else np.nan,
        "min_year": df["Year"].min() if "Year" in df.columns and len(df) else np.nan,
        "max_year": df["Year"].max() if "Year" in df.columns and len(df) else np.nan,
    }

In [6]:
# ------------------------------------------------------
# OECD REST SOURCE CONFIGURATION
# ------------------------------------------------------
# No country filters are used here.
# Only indicator-definition filters/query keys are used.
#
# GDP growth is not here because the OECD REST query returned NoResultsFound.
# Productivity is not here because the OECD REST productivity route failed.
# Both are acquired via DBnomics OECD mirror below.

OECD_REST_DATASETS = [
    {
        "file_name": "OECD_RD_GDP_2002_2025.csv",
        "dataset_label": "OECD_RD_GDP",
        "agency_id": "OECD.STI.STP",
        "dataflow_id": "DSD_MSTI@DF_MSTI",
        "sdmx_query_key": "all",
        "filters": {
            "UNIT_MEASURE": "PT_B1GQ",
            "MEASURE": "G",
        },
        "definition_note": "R&D expenditure as percentage of GDP.",
        "required": True,
    },
    {
        "file_name": "OECD_Inflation_CPI_2002_2025.csv",
        "dataset_label": "OECD_Inflation_CPI",
        "agency_id": "OECD.SDD.TPS",
        "dataflow_id": "DSD_PRICES@DF_PRICES_ALL",
        "sdmx_query_key": ".A.N.CPI.._T.N.GY",
        "filters": {},
        "definition_note": "Consumer price inflation, annual growth rate.",
        "required": True,
    },
    {
        "file_name": "OECD_Unemployment_Rate_2002_2025.csv",
        "dataset_label": "OECD_Unemployment_Rate",
        "agency_id": "OECD.ELS.SAE",
        "dataflow_id": "DSD_LFS@DF_LFS_INDIC",
        "sdmx_query_key": "all",
        "filters": {
            "MEASURE": "UNE_RATE",
            "UNIT_MEASURE": "PT_LF_SUB",
            "SEX": "_T",
            "AGE": "_T",
        },
        "definition_note": "Unemployment rate, total sex, total age, percentage of labour force.",
        "required": True,
    },
    {
        "file_name": "OECD_Tertiary_Education_2002_2025.csv",
        "dataset_label": "OECD_Tertiary_Education",
        "agency_id": "OECD.EDU.IMEP",
        "dataflow_id": "DSD_EAG_LSO_EA@DF_LSO_NEAC_DISTR_EA",
        "sdmx_query_key": "._T.Y25T64.ISCED11A_5T8..........OBS...A",
        "filters": {},
        "definition_note": "Adult tertiary attainment: age 25-64, total sex, tertiary education. Not enrolment/student-flow data.",
        "required": True,
    },
]

pd.DataFrame(OECD_REST_DATASETS)

,file_name,dataset_label,agency_id,dataflow_id,sdmx_query_key,filters,definition_note,required
0,OECD_RD_GDP_2002_2025.csv,OECD_RD_GDP,OECD.STI.STP,DSD_MSTI@DF_MSTI,all,"{'UNIT_MEASURE': 'PT_B1GQ', 'MEASURE': 'G'}",R&D expenditure as percentage of GDP.,True
1,OECD_Inflation_CPI_2002_2025.csv,OECD_Inflation_CPI,OECD.SDD.TPS,DSD_PRICES@DF_PRICES_ALL,.A.N.CPI.._T.N.GY,{},"Consumer price inflation, annual growth rate.",True
2,OECD_Unemployment_Rate_2002_2025.csv,OECD_Unemployment_Rate,OECD.ELS.SAE,DSD_LFS@DF_LFS_INDIC,all,"{'MEASURE': 'UNE_RATE', 'UNIT_MEASURE': 'PT_LF...","Unemployment rate, total sex, total age, perce...",True
3,OECD_Tertiary_Education_2002_2025.csv,OECD_Tertiary_Education,OECD.EDU.IMEP,DSD_EAG_LSO_EA@DF_LSO_NEAC_DISTR_EA,._T.Y25T64.ISCED11A_5T8..........OBS...A,{},"Adult tertiary attainment: age 25-64, total se...",True


In [7]:
# ------------------------------------------------------
# OECD REST DOWNLOADER
# ------------------------------------------------------

def build_oecd_url(agency_id, dataflow_id, query_key):
    return (
        f"{OECD_BASE_URL}/"
        f"{agency_id},{dataflow_id},{OECD_VERSION}/"
        f"{query_key}"
    )


def download_oecd_dataset(dataset):
    dataset_label = dataset["dataset_label"]
    file_name = dataset["file_name"]
    output_path = RAW_FILES_DIR / file_name

    print("=" * 80)
    print(f"Processing OECD dataset: {file_name}")

    url = build_oecd_url(
        agency_id=dataset["agency_id"],
        dataflow_id=dataset["dataflow_id"],
        query_key=dataset["sdmx_query_key"],
    )

    params = {
        "startPeriod": str(START_YEAR),
        "endPeriod": str(END_YEAR),
        "dimensionAtObservation": "AllDimensions",
        "format": "csvfilewithlabels",
    }

    response = request_with_retry(
        url=url,
        params=params,
        headers=OECD_HEADERS,
    )

    if response.status_code != 200:
        raise RuntimeError(f"HTTP {response.status_code}: {response.text[:500]}")

    df_raw = pd.read_csv(io.StringIO(response.text), low_memory=False)

    print(f"  -> Raw rows returned: {len(df_raw)}")

    required_cols = {"REF_AREA", "TIME_PERIOD", "OBS_VALUE"}
    missing_required = required_cols - set(df_raw.columns)

    if missing_required:
        raise ValueError(f"{dataset_label}: missing columns {missing_required}")

    df_filtered = df_raw.copy()

    filter_audit_rows = []

    for col, expected_value in dataset.get("filters", {}).items():
        if col not in df_filtered.columns:
            raise ValueError(
                f"{dataset_label}: required filter column {col} is missing. "
                f"Available columns: {df_filtered.columns.tolist()}"
            )

        before = len(df_filtered)
        df_filtered = df_filtered[
            df_filtered[col].astype(str).str.strip().eq(str(expected_value))
        ].copy()
        after = len(df_filtered)

        print(f"  -> Filter {col} == {expected_value}: {before} → {after} rows")

        filter_audit_rows.append({
            "dataset_label": dataset_label,
            "column": col,
            "expected_value": expected_value,
            "rows_before": before,
            "rows_after": after,
        })

    if df_filtered.empty:
        raise ValueError(f"{dataset_label}: no rows after filtering.")

    dimension_audit_rows = []

    skip_cols = {"OBS_VALUE", "OBS_STATUS", "TIME_PERIOD", "REF_AREA"}

    for col in df_raw.columns:
        if col in skip_cols:
            continue

        values = df_raw[col].dropna().astype(str).unique().tolist()

        dimension_audit_rows.append({
            "dataset_label": dataset_label,
            "column": col,
            "unique_count_raw": len(values),
            "sample_values_raw": "; ".join(sorted(values)[:100]),
        })

    pd.DataFrame(dimension_audit_rows).to_csv(
        DIAGNOSTICS_DIR / f"oecd_dimension_audit__{safe_file_token(dataset_label)}.csv",
        index=False,
    )

    if filter_audit_rows:
        pd.DataFrame(filter_audit_rows).to_csv(
            DIAGNOSTICS_DIR / f"oecd_filter_audit__{safe_file_token(dataset_label)}.csv",
            index=False,
        )

    if "UNIT_MULT" in df_raw.columns:
        unit_mult_audit = pd.DataFrame([{
            "dataset_label": dataset_label,
            "has_unit_mult": True,
            "unique_unit_mult_values_raw": json.dumps(
                sorted(
                    pd.to_numeric(df_raw["UNIT_MULT"], errors="coerce")
                    .dropna()
                    .unique()
                    .tolist()
                )
            ),
            "note": "UNIT_MULT logged only; not applied.",
        }])
    else:
        unit_mult_audit = pd.DataFrame([{
            "dataset_label": dataset_label,
            "has_unit_mult": False,
            "unique_unit_mult_values_raw": "",
            "note": "UNIT_MULT column not present.",
        }])

    unit_mult_audit.to_csv(
        DIAGNOSTICS_DIR / f"oecd_unit_mult_audit__{safe_file_token(dataset_label)}.csv",
        index=False,
    )

    df_tidy = pd.DataFrame({
        "Country": df_filtered["REF_AREA"].astype(str).str.strip().str.upper(),
        "Year": parse_annual_year(df_filtered["TIME_PERIOD"], dataset_label),
        "Value": pd.to_numeric(df_filtered["OBS_VALUE"], errors="coerce"),
    })

    df_tidy = df_tidy.dropna(subset=["Value"]).copy()

    df_tidy = validate_tidy_country_year_value(
        df_tidy,
        dataset_name=dataset_label,
        stop_on_duplicates=True,
    )

    write_tidy_csv(df_tidy, output_path, overwrite=OVERWRITE_RAW_FILES)

    print(f"  -> Success: {len(df_tidy)} rows saved to {output_path}")

    add_acquisition_log(
        dataset_label=dataset_label,
        file_name=file_name,
        status="success",
        source_route="OECD_REST",
        rows_raw=len(df_raw),
        rows_saved=len(df_tidy),
        countries=df_tidy["Country"].nunique(),
        min_year=df_tidy["Year"].min(),
        max_year=df_tidy["Year"].max(),
        output_file=output_path,
        note=dataset.get("definition_note", ""),
        url=url,
    )

    return df_tidy

In [8]:
# ------------------------------------------------------
# RUN OECD REST DOWNLOADS
# ------------------------------------------------------

oecd_outputs = {}

for dataset in OECD_REST_DATASETS:
    try:
        df_oecd = download_oecd_dataset(dataset)
        oecd_outputs[dataset["dataset_label"]] = df_oecd

    except Exception as e:
        dataset_label = dataset.get("dataset_label", dataset.get("file_name", "unknown_dataset"))
        required = dataset.get("required", True)

        print("\n" + "!" * 80)
        print(f"OECD dataset failed: {dataset_label}")
        print("Required:", required)
        print("Error:", e)
        print("!" * 80 + "\n")

        add_failed_download(
            dataset_label=dataset_label,
            source_route="OECD_REST",
            error=e,
        )

        if required and STOP_ON_REQUIRED_FAILURE:
            raise
        else:
            print(f"Continuing after failure. This must be reviewed later: {dataset_label}")

print("\nOECD REST downloads attempted.")
print("Successful OECD datasets:", list(oecd_outputs.keys()))

Processing OECD dataset: OECD_RD_GDP_2002_2025.csv
  -> Raw rows returned: 139178
  -> Filter UNIT_MEASURE == PT_B1GQ: 139178 → 9097 rows
  -> Filter MEASURE == G: 9097 → 1044 rows
  -> Success: 1044 rows saved to D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Raw\Raw_Files\OECD_RD_GDP_2002_2025.csv
Processing OECD dataset: OECD_Inflation_CPI_2002_2025.csv
  -> Raw rows returned: 1257
  -> Success: 1257 rows saved to D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Raw\Raw_Files\OECD_Inflation_CPI_2002_2025.csv
Processing OECD dataset: OECD_Unemployment_Rate_2002_2025.csv
  -> Raw rows returned: 332574
  -> Filter MEASURE == UNE_RATE: 332574 → 109755 rows
  -> Filter UNIT_MEASURE == PT_LF_SUB: 109755 → 109755 rows
  -> Filter SEX == _T: 109755 → 36642 rows
  -> Filter AGE == _T: 36642 → 1338 rows
  -> Success: 1338 rows saved to D:\Mi

In [9]:
# ------------------------------------------------------
# DBNOMICS GENERIC HELPERS
# ------------------------------------------------------

def build_dbnomics_series_url(provider, dataset, series_code=None):
    provider_q = quote(provider, safe="")
    dataset_q = quote(dataset, safe="")

    if series_code is None:
        return f"{DBNOMICS_BASE_URL}/series/{provider_q}/{dataset_q}"

    series_q = quote(series_code, safe="")
    return f"{DBNOMICS_BASE_URL}/series/{provider_q}/{dataset_q}/{series_q}"


def parse_dbnomics_payload(payload, country, series_code):
    rows = []

    if not isinstance(payload, dict):
        return pd.DataFrame(columns=["Country", "Year", "Value", "Series_Code"])

    series_obj = payload.get("series", {})
    docs = series_obj.get("docs", [])

    if not docs:
        return pd.DataFrame(columns=["Country", "Year", "Value", "Series_Code"])

    doc = docs[0]
    values = doc.get("values")

    if isinstance(values, dict):
        for period, value in values.items():
            if value is None:
                continue

            try:
                year = int(str(period))
                val = float(value)
            except Exception:
                continue

            rows.append({
                "Country": country,
                "Year": year,
                "Value": val,
                "Series_Code": series_code,
            })

    elif "period" in doc and "value" in doc:
        periods = doc.get("period", [])
        vals = doc.get("value", [])

        for period, value in zip(periods, vals):
            if value is None:
                continue

            try:
                year = int(str(period))
                val = float(value)
            except Exception:
                continue

            rows.append({
                "Country": country,
                "Year": year,
                "Value": val,
                "Series_Code": series_code,
            })

    elif isinstance(values, list):
        for item in values:
            if not isinstance(item, dict):
                continue

            period = item.get("period") or item.get("time_period") or item.get("date")
            value = item.get("value")

            if value is None:
                continue

            try:
                year = int(str(period))
                val = float(value)
            except Exception:
                continue

            rows.append({
                "Country": country,
                "Year": year,
                "Value": val,
                "Series_Code": series_code,
            })

    return pd.DataFrame(rows)


def find_strings_containing(obj, token):
    found = []

    if isinstance(obj, dict):
        for value in obj.values():
            found.extend(find_strings_containing(value, token))

    elif isinstance(obj, list):
        for value in obj:
            found.extend(find_strings_containing(value, token))

    elif isinstance(obj, str):
        text = obj.strip()

        if token in text:
            if "/" in text:
                text = text.split("/")[-1]
            found.append(text)

    return found


def fetch_dbnomics_series(provider, dataset, country, series_code):
    url = build_dbnomics_series_url(
        provider=provider,
        dataset=dataset,
        series_code=series_code,
    )

    response = request_with_retry(
        url=url,
        params={"observations": "1"},
        headers=DBNOMICS_HEADERS,
    )

    status_row = {
        "country": country,
        "series_code": series_code,
        "url": url,
        "status_code": response.status_code,
        "status": "",
        "rows": 0,
        "min_year": np.nan,
        "max_year": np.nan,
        "error": "",
    }

    if response.status_code != 200:
        status_row["status"] = "http_failed"
        status_row["error"] = response.text[:500]
        return status_row, pd.DataFrame(columns=["Country", "Year", "Value", "Series_Code"])

    try:
        payload = response.json()
    except Exception as e:
        status_row["status"] = "json_parse_failed"
        status_row["error"] = str(e)
        return status_row, pd.DataFrame(columns=["Country", "Year", "Value", "Series_Code"])

    df_series = parse_dbnomics_payload(
        payload=payload,
        country=country,
        series_code=series_code,
    )

    df_series = df_series[
        (df_series["Year"] >= START_YEAR)
        & (df_series["Year"] <= END_YEAR)
    ].copy()

    status_row["rows"] = len(df_series)

    if df_series.empty:
        status_row["status"] = "empty_series"
    else:
        status_row["status"] = "success"
        status_row["min_year"] = df_series["Year"].min()
        status_row["max_year"] = df_series["Year"].max()

    return status_row, df_series

In [10]:
# ------------------------------------------------------
# PRODUCTIVITY VIA DBNOMICS OECD MIRROR
# ------------------------------------------------------
# Target:
# GDP per hour worked, total activities, USD PPP per hour, constant prices.
#
# Previous issue:
# Old productivity extraction matched PRICE_BASE = V current prices.
# Corrected route uses PRICE_BASE = Q constant prices.
#
# Series pattern:
# COUNTRY_OR_GEO.A.GDPHRS._T.USD_PPP_H.Q._Z._Z._Z

PRODUCTIVITY_FILE_NAME = "OECD_Productivity_GDP_per_Hour_2002_2025.csv"
PRODUCTIVITY_OUTPUT_FILE = RAW_FILES_DIR / PRODUCTIVITY_FILE_NAME

DBNOMICS_PRODUCTIVITY_PROVIDER = "OECD"
DBNOMICS_PRODUCTIVITY_DATASET = "DSD_PDB@DF_PDB_LV"

PRODUCTIVITY_SUFFIX = ".A.GDPHRS._T.USD_PPP_H.Q._Z._Z._Z"
PRODUCTIVITY_PATTERN = f"*{PRODUCTIVITY_SUFFIX}"


def get_productivity_doc_series_code(doc):
    direct_candidates = [
        doc.get("code"),
        doc.get("series_code"),
        doc.get("id"),
        doc.get("series_id"),
        doc.get("@id"),
    ]

    for candidate in direct_candidates:
        if not candidate:
            continue

        candidate = str(candidate).strip()

        if "/" in candidate:
            candidate = candidate.split("/")[-1]

        if candidate.endswith(PRODUCTIVITY_SUFFIX):
            return candidate

    recursive_candidates = find_strings_containing(doc, ".A.GDPHRS.")

    for candidate in recursive_candidates:
        candidate = str(candidate).strip()

        if candidate.endswith(PRODUCTIVITY_SUFFIX):
            return candidate

    return None


def discover_productivity_series():
    print("=" * 80)
    print("Discovering productivity series from DBnomics")
    print("Pattern:", PRODUCTIVITY_PATTERN)

    url = build_dbnomics_series_url(
        provider=DBNOMICS_PRODUCTIVITY_PROVIDER,
        dataset=DBNOMICS_PRODUCTIVITY_DATASET,
        series_code=None,
    )

    all_docs = []
    limit = 500
    offset = 0

    while True:
        response = request_with_retry(
            url=url,
            params={"limit": limit, "offset": offset},
            headers=DBNOMICS_HEADERS,
        )

        if response.status_code != 200:
            error_path = DIAGNOSTICS_DIR / "dbnomics_productivity_series_discovery_error.txt"
            error_path.write_text(response.text[:5000], encoding="utf-8")
            raise RuntimeError(f"DBnomics productivity discovery failed. HTTP {response.status_code}")

        payload = response.json()

        if offset == 0:
            preview_path = DIAGNOSTICS_DIR / "dbnomics_productivity_series_discovery_payload_preview.json"
            preview_path.write_text(
                json.dumps(payload, indent=2, ensure_ascii=False)[:200000],
                encoding="utf-8",
            )

        docs = payload.get("series", {}).get("docs", [])

        if not docs:
            break

        all_docs.extend(docs)
        print(f"  -> Retrieved metadata docs: {len(all_docs)}")

        if len(docs) < limit:
            break

        offset += limit

        if offset > 10000:
            raise RuntimeError("Productivity discovery pagination exceeded 10,000 docs.")

    discovered_rows = []

    for doc in all_docs:
        series_code = get_productivity_doc_series_code(doc)

        if not series_code:
            continue

        parts = series_code.split(".")

        if len(parts) < 9:
            continue

        discovered_rows.append({
            "Country": parts[0],
            "Series_Code": series_code,
            "Series_Name": doc.get("name") or doc.get("title") or "",
        })

    discovered = pd.DataFrame(discovered_rows)

    if discovered.empty:
        raise ValueError("No productivity series matched the target pattern.")

    discovered = (
        discovered
        .drop_duplicates(subset=["Country", "Series_Code"])
        .sort_values(["Country", "Series_Code"])
        .reset_index(drop=True)
    )

    discovered.to_csv(
        DIAGNOSTICS_DIR / "dbnomics_productivity_discovered_series_codes.csv",
        index=False,
    )

    print("Discovered productivity series:", len(discovered))
    print("Countries/geographies:", discovered["Country"].nunique())

    display(discovered.head(20))

    return discovered


def download_productivity_dbnomics():
    print("=" * 80)
    print("Processing productivity via DBnomics OECD mirror")
    print("Country filter: NONE")

    discovered = discover_productivity_series()

    status_rows = []
    frames = []

    for _, row in discovered.iterrows():
        status_row, df_country = fetch_dbnomics_series(
            provider=DBNOMICS_PRODUCTIVITY_PROVIDER,
            dataset=DBNOMICS_PRODUCTIVITY_DATASET,
            country=row["Country"],
            series_code=row["Series_Code"],
        )

        status_rows.append(status_row)

        if not df_country.empty:
            frames.append(df_country)

    status_df = pd.DataFrame(status_rows)

    status_df.to_csv(
        DIAGNOSTICS_DIR / "dbnomics_productivity_country_status.csv",
        index=False,
    )

    if frames:
        rich = pd.concat(frames, ignore_index=True)
    else:
        rich = pd.DataFrame(columns=["Country", "Year", "Value", "Series_Code"])

    rich.to_csv(
        DIAGNOSTICS_DIR / "dbnomics_productivity_rich_output.csv",
        index=False,
    )

    tidy = rich[["Country", "Year", "Value"]].copy()

    tidy = validate_tidy_country_year_value(
        tidy,
        dataset_name="OECD_Productivity_GDP_per_Hour_DBNomics_Q",
        stop_on_duplicates=True,
    )

    write_tidy_csv(tidy, PRODUCTIVITY_OUTPUT_FILE, overwrite=OVERWRITE_RAW_FILES)

    sanity = pd.DataFrame([{
        "dataset_label": "OECD_Productivity_GDP_per_Hour",
        "source_route": "DBNOMICS_OECD_MIRROR",
        "series_pattern": PRODUCTIVITY_PATTERN,
        "country_filter": "none_all_available_series",
        "price_base": "Q_constant_prices",
        "rows_saved": len(tidy),
        "countries": tidy["Country"].nunique(),
        "min_year": tidy["Year"].min(),
        "max_year": tidy["Year"].max(),
        "min_value": tidy["Value"].min(),
        "median_value": tidy["Value"].median(),
        "max_value": tidy["Value"].max(),
    }])

    sanity.to_csv(
        DIAGNOSTICS_DIR / "dbnomics_productivity_sanity_check.csv",
        index=False,
    )

    add_acquisition_log(
        dataset_label="OECD_Productivity_GDP_per_Hour",
        file_name=PRODUCTIVITY_FILE_NAME,
        status="success",
        source_route="DBNOMICS_OECD_MIRROR",
        rows_raw=len(rich),
        rows_saved=len(tidy),
        countries=tidy["Country"].nunique(),
        min_year=tidy["Year"].min(),
        max_year=tidy["Year"].max(),
        output_file=PRODUCTIVITY_OUTPUT_FILE,
        note="GDP per hour worked, total activities, USD PPP per hour, constant prices. All available countries/geographies pulled.",
        url="DBnomics OECD/DSD_PDB@DF_PDB_LV",
    )

    print(f"  -> Success: {len(tidy)} rows saved to {PRODUCTIVITY_OUTPUT_FILE}")

    display(sanity)

    return tidy


try:
    productivity_output = download_productivity_dbnomics()

except Exception as e:
    print("\n" + "!" * 80)
    print("Productivity acquisition failed.")
    print("Error:", e)
    print("!" * 80 + "\n")

    add_failed_download(
        dataset_label="OECD_Productivity_GDP_per_Hour",
        source_route="DBNOMICS_OECD_MIRROR",
        error=e,
    )

    if STOP_ON_REQUIRED_FAILURE:
        raise

Processing productivity via DBnomics OECD mirror
Country filter: NONE
Discovering productivity series from DBnomics
Pattern: *.A.GDPHRS._T.USD_PPP_H.Q._Z._Z._Z
  -> Retrieved metadata docs: 500
  -> Retrieved metadata docs: 871
Discovered productivity series: 44
Countries/geographies: 44


,Country,Series_Code,Series_Name
0,AUS,AUS.A.GDPHRS._T.USD_PPP_H.Q._Z._Z._Z,
1,AUT,AUT.A.GDPHRS._T.USD_PPP_H.Q._Z._Z._Z,
2,BEL,BEL.A.GDPHRS._T.USD_PPP_H.Q._Z._Z._Z,
3,BGR,BGR.A.GDPHRS._T.USD_PPP_H.Q._Z._Z._Z,
4,CAN,CAN.A.GDPHRS._T.USD_PPP_H.Q._Z._Z._Z,
5,CHE,CHE.A.GDPHRS._T.USD_PPP_H.Q._Z._Z._Z,
6,CHL,CHL.A.GDPHRS._T.USD_PPP_H.Q._Z._Z._Z,
7,COL,COL.A.GDPHRS._T.USD_PPP_H.Q._Z._Z._Z,
8,CRI,CRI.A.GDPHRS._T.USD_PPP_H.Q._Z._Z._Z,
9,CZE,CZE.A.GDPHRS._T.USD_PPP_H.Q._Z._Z._Z,


  -> Success: 991 rows saved to D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Raw\Raw_Files\OECD_Productivity_GDP_per_Hour_2002_2025.csv


,dataset_label,source_route,series_pattern,country_filter,price_base,rows_saved,countries,min_year,max_year,min_value,median_value,max_value
0,OECD_Productivity_GDP_per_Hour,DBNOMICS_OECD_MIRROR,*.A.GDPHRS._T.USD_PPP_H.Q._Z._Z._Z,none_all_available_series,Q_constant_prices,991,44,2002,2024,11.7448,53.7709,142.9044


In [11]:
# ------------------------------------------------------
# GDP GROWTH VIA DBNOMICS OECD MIRROR
# ------------------------------------------------------
# Target:
# Real gross domestic product growth rate, percentage change.
#
# Series pattern:
# A.COUNTRY_OR_GEO.B1GQ_R_GR.PC.NAAG_I

GDP_GROWTH_FILE_NAME = "OECD_GDP_Growth_2002_2025.csv"
GDP_GROWTH_OUTPUT_FILE = RAW_FILES_DIR / GDP_GROWTH_FILE_NAME

DBNOMICS_GDP_PROVIDER = "OECD"
DBNOMICS_GDP_DATASET = "DSD_NAAG@DF_NAAG_I"

GDP_GROWTH_PREFIX = "A."
GDP_GROWTH_SUFFIX = ".B1GQ_R_GR.PC.NAAG_I"
GDP_GROWTH_PATTERN = f"{GDP_GROWTH_PREFIX}*{GDP_GROWTH_SUFFIX}"


def get_gdp_growth_doc_series_code(doc):
    direct_candidates = [
        doc.get("code"),
        doc.get("series_code"),
        doc.get("id"),
        doc.get("series_id"),
        doc.get("@id"),
    ]

    for candidate in direct_candidates:
        if not candidate:
            continue

        candidate = str(candidate).strip()

        if "/" in candidate:
            candidate = candidate.split("/")[-1]

        if candidate.startswith(GDP_GROWTH_PREFIX) and candidate.endswith(GDP_GROWTH_SUFFIX):
            return candidate

    recursive_candidates = find_strings_containing(doc, "B1GQ_R_GR")

    for candidate in recursive_candidates:
        candidate = str(candidate).strip()

        if candidate.startswith(GDP_GROWTH_PREFIX) and candidate.endswith(GDP_GROWTH_SUFFIX):
            return candidate

    return None


def discover_gdp_growth_series():
    print("=" * 80)
    print("Discovering GDP growth series from DBnomics")
    print("Pattern:", GDP_GROWTH_PATTERN)

    url = build_dbnomics_series_url(
        provider=DBNOMICS_GDP_PROVIDER,
        dataset=DBNOMICS_GDP_DATASET,
        series_code=None,
    )

    all_docs = []
    limit = 500
    offset = 0

    while True:
        response = request_with_retry(
            url=url,
            params={"limit": limit, "offset": offset},
            headers=DBNOMICS_HEADERS,
        )

        if response.status_code != 200:
            error_path = DIAGNOSTICS_DIR / "dbnomics_gdp_growth_series_discovery_error.txt"
            error_path.write_text(response.text[:5000], encoding="utf-8")
            raise RuntimeError(f"DBnomics GDP growth discovery failed. HTTP {response.status_code}")

        payload = response.json()

        if offset == 0:
            preview_path = DIAGNOSTICS_DIR / "dbnomics_gdp_growth_series_discovery_payload_preview.json"
            preview_path.write_text(
                json.dumps(payload, indent=2, ensure_ascii=False)[:200000],
                encoding="utf-8",
            )

        docs = payload.get("series", {}).get("docs", [])

        if not docs:
            break

        all_docs.extend(docs)
        print(f"  -> Retrieved metadata docs: {len(all_docs)}")

        if len(docs) < limit:
            break

        offset += limit

        if offset > 10000:
            raise RuntimeError("GDP growth discovery pagination exceeded 10,000 docs.")

    discovered_rows = []

    for doc in all_docs:
        series_code = get_gdp_growth_doc_series_code(doc)

        if not series_code:
            continue

        parts = series_code.split(".")

        # Expected: A.COUNTRY.B1GQ_R_GR.PC.NAAG_I
        if len(parts) != 5:
            continue

        discovered_rows.append({
            "Country": parts[1],
            "Series_Code": series_code,
            "Series_Name": doc.get("name") or doc.get("title") or "",
        })

    discovered = pd.DataFrame(discovered_rows)

    if discovered.empty:
        raise ValueError("No GDP growth series matched the target pattern.")

    discovered = (
        discovered
        .drop_duplicates(subset=["Country", "Series_Code"])
        .sort_values(["Country", "Series_Code"])
        .reset_index(drop=True)
    )

    discovered.to_csv(
        DIAGNOSTICS_DIR / "dbnomics_gdp_growth_discovered_series_codes.csv",
        index=False,
    )

    print("Discovered GDP growth series:", len(discovered))
    print("Countries/geographies:", discovered["Country"].nunique())

    display(discovered.head(20))

    return discovered


def download_gdp_growth_dbnomics():
    print("=" * 80)
    print("Processing GDP growth via DBnomics OECD mirror")
    print("Country filter: NONE")

    discovered = discover_gdp_growth_series()

    status_rows = []
    frames = []

    for _, row in discovered.iterrows():
        status_row, df_country = fetch_dbnomics_series(
            provider=DBNOMICS_GDP_PROVIDER,
            dataset=DBNOMICS_GDP_DATASET,
            country=row["Country"],
            series_code=row["Series_Code"],
        )

        status_rows.append(status_row)

        if not df_country.empty:
            frames.append(df_country)

    status_df = pd.DataFrame(status_rows)

    status_df.to_csv(
        DIAGNOSTICS_DIR / "dbnomics_gdp_growth_country_status.csv",
        index=False,
    )

    if frames:
        rich = pd.concat(frames, ignore_index=True)
    else:
        rich = pd.DataFrame(columns=["Country", "Year", "Value", "Series_Code"])

    rich.to_csv(
        DIAGNOSTICS_DIR / "dbnomics_gdp_growth_rich_output.csv",
        index=False,
    )

    tidy = rich[["Country", "Year", "Value"]].copy()

    tidy = validate_tidy_country_year_value(
        tidy,
        dataset_name="OECD_GDP_Growth_DBNomics",
        stop_on_duplicates=True,
    )

    write_tidy_csv(tidy, GDP_GROWTH_OUTPUT_FILE, overwrite=OVERWRITE_RAW_FILES)

    sanity = pd.DataFrame([{
        "dataset_label": "OECD_GDP_Growth",
        "source_route": "DBNOMICS_OECD_MIRROR",
        "series_pattern": GDP_GROWTH_PATTERN,
        "country_filter": "none_all_available_series",
        "rows_saved": len(tidy),
        "countries": tidy["Country"].nunique(),
        "min_year": tidy["Year"].min(),
        "max_year": tidy["Year"].max(),
        "min_value": tidy["Value"].min(),
        "median_value": tidy["Value"].median(),
        "max_value": tidy["Value"].max(),
    }])

    sanity.to_csv(
        DIAGNOSTICS_DIR / "dbnomics_gdp_growth_sanity_check.csv",
        index=False,
    )

    add_acquisition_log(
        dataset_label="OECD_GDP_Growth",
        file_name=GDP_GROWTH_FILE_NAME,
        status="success",
        source_route="DBNOMICS_OECD_MIRROR",
        rows_raw=len(rich),
        rows_saved=len(tidy),
        countries=tidy["Country"].nunique(),
        min_year=tidy["Year"].min(),
        max_year=tidy["Year"].max(),
        output_file=GDP_GROWTH_OUTPUT_FILE,
        note="Real gross domestic product growth rate, percentage change. All available countries/geographies pulled.",
        url="DBnomics OECD/DSD_NAAG@DF_NAAG_I",
    )

    print(f"  -> Success: {len(tidy)} rows saved to {GDP_GROWTH_OUTPUT_FILE}")

    display(sanity)

    return tidy


try:
    gdp_growth_output = download_gdp_growth_dbnomics()

except Exception as e:
    print("\n" + "!" * 80)
    print("GDP growth acquisition failed.")
    print("Error:", e)
    print("!" * 80 + "\n")

    add_failed_download(
        dataset_label="OECD_GDP_Growth",
        source_route="DBNOMICS_OECD_MIRROR",
        error=e,
    )

    if STOP_ON_REQUIRED_FAILURE:
        raise

Processing GDP growth via DBnomics OECD mirror
Country filter: NONE
Discovering GDP growth series from DBnomics
Pattern: A.*.B1GQ_R_GR.PC.NAAG_I
  -> Retrieved metadata docs: 420
Discovered GDP growth series: 47
Countries/geographies: 47


,Country,Series_Code,Series_Name
0,AUS,A.AUS.B1GQ_R_GR.PC.NAAG_I,
1,AUT,A.AUT.B1GQ_R_GR.PC.NAAG_I,
2,BEL,A.BEL.B1GQ_R_GR.PC.NAAG_I,
3,BRA,A.BRA.B1GQ_R_GR.PC.NAAG_I,
4,CAN,A.CAN.B1GQ_R_GR.PC.NAAG_I,
5,CHE,A.CHE.B1GQ_R_GR.PC.NAAG_I,
6,CHL,A.CHL.B1GQ_R_GR.PC.NAAG_I,
7,CHN,A.CHN.B1GQ_R_GR.PC.NAAG_I,
8,COL,A.COL.B1GQ_R_GR.PC.NAAG_I,
9,CRI,A.CRI.B1GQ_R_GR.PC.NAAG_I,


  -> Success: 1115 rows saved to D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Raw\Raw_Files\OECD_GDP_Growth_2002_2025.csv


,dataset_label,source_route,series_pattern,country_filter,rows_saved,countries,min_year,max_year,min_value,median_value,max_value
0,OECD_GDP_Growth,DBNOMICS_OECD_MIRROR,A.*.B1GQ_R_GR.PC.NAAG_I,none_all_available_series,1115,47,2002,2025,-16.0400,2.4609,24.6240


In [12]:
# ------------------------------------------------------
# WORLD BANK DOWNLOADER
# ------------------------------------------------------
# World Bank pulls country/all.
# No project-specific country filtering is applied here.
#
# Energy imports are kept raw.
# Negative values are meaningful because they indicate net exporters.

WORLD_BANK_DATASETS = [
    {
        "file_name": "WorldBank_Public_Debt_GDP_2002_2025.csv",
        "dataset_label": "WorldBank_Public_Debt_GDP",
        "indicator_code": "GC.DOD.TOTL.GD.ZS",
        "definition_note": "Central government debt, total (% of GDP).",
        "required": True,
    },
    {
        "file_name": "WorldBank_Energy_Import_Dependency_Raw_2002_2025.csv",
        "dataset_label": "WorldBank_Energy_Import_Dependency_Raw",
        "indicator_code": "EG.IMP.CONS.ZS",
        "definition_note": "Energy imports, net (% of energy use). Raw values preserved; negative values are meaningful.",
        "required": True,
    },
]


def download_world_bank_indicator(dataset):
    dataset_label = dataset["dataset_label"]
    indicator_code = dataset["indicator_code"]
    file_name = dataset["file_name"]
    output_path = RAW_FILES_DIR / file_name

    print("=" * 80)
    print(f"Processing World Bank dataset: {file_name}")

    url = f"{WORLD_BANK_BASE_URL}/country/all/indicator/{indicator_code}"

    params = {
        "format": "json",
        "per_page": 20000,
        "date": f"{START_YEAR}:{END_YEAR}",
    }

    response = request_with_retry(
        url=url,
        params=params,
        headers={"User-Agent": "EconomicResilienceProject/1.0 (Python/Requests)"},
    )

    if response.status_code != 200:
        raise RuntimeError(f"HTTP {response.status_code}: {response.text[:500]}")

    payload = response.json()

    if not isinstance(payload, list) or len(payload) < 2:
        raise ValueError(f"{dataset_label}: unexpected World Bank response structure.")

    data = payload[1]

    rows = []

    for item in data:
        value = item.get("value")

        if value is None:
            continue

        country_info = item.get("country", {})
        countryiso3code = item.get("countryiso3code")

        if not countryiso3code:
            continue

        try:
            year = int(item.get("date"))
            val = float(value)
        except Exception:
            continue

        rows.append({
            "Country": str(countryiso3code).strip().upper(),
            "Year": year,
            "Value": val,
            "Country_Name": country_info.get("value"),
            "Indicator_Code": indicator_code,
            "Indicator_Name": item.get("indicator", {}).get("value"),
        })

    df_rich = pd.DataFrame(rows)

    df_rich.to_csv(
        DIAGNOSTICS_DIR / f"worldbank_rich_output__{safe_file_token(dataset_label)}.csv",
        index=False,
    )

    df_tidy = df_rich[["Country", "Year", "Value"]].copy()

    df_tidy = validate_tidy_country_year_value(
        df_tidy,
        dataset_name=dataset_label,
        stop_on_duplicates=True,
    )

    write_tidy_csv(df_tidy, output_path, overwrite=OVERWRITE_RAW_FILES)

    print(f"  -> Success: {len(df_tidy)} rows saved to {output_path}")

    add_acquisition_log(
        dataset_label=dataset_label,
        file_name=file_name,
        status="success",
        source_route="WORLD_BANK_API",
        rows_raw=len(df_rich),
        rows_saved=len(df_tidy),
        countries=df_tidy["Country"].nunique(),
        min_year=df_tidy["Year"].min(),
        max_year=df_tidy["Year"].max(),
        output_file=output_path,
        note=dataset.get("definition_note", ""),
        url=url,
    )

    return df_tidy


worldbank_outputs = {}

for dataset in WORLD_BANK_DATASETS:
    try:
        df_wb = download_world_bank_indicator(dataset)
        worldbank_outputs[dataset["dataset_label"]] = df_wb

    except Exception as e:
        dataset_label = dataset.get("dataset_label", dataset.get("file_name", "unknown_worldbank_dataset"))
        required = dataset.get("required", True)

        print("\n" + "!" * 80)
        print(f"World Bank dataset failed: {dataset_label}")
        print("Required:", required)
        print("Error:", e)
        print("!" * 80 + "\n")

        add_failed_download(
            dataset_label=dataset_label,
            source_route="WORLD_BANK_API",
            error=e,
        )

        if required and STOP_ON_REQUIRED_FAILURE:
            raise

Processing World Bank dataset: WorldBank_Public_Debt_GDP_2002_2025.csv
  -> Success: 1160 rows saved to D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Raw\Raw_Files\WorldBank_Public_Debt_GDP_2002_2025.csv
Processing World Bank dataset: WorldBank_Energy_Import_Dependency_Raw_2002_2025.csv
  -> Success: 3798 rows saved to D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Raw\Raw_Files\WorldBank_Energy_Import_Dependency_Raw_2002_2025.csv


In [13]:
# ------------------------------------------------------
# EUROSTAT JSON-STAT PARSER
# ------------------------------------------------------

def parse_eurostat_jsonstat(payload, value_name="Value"):
    ids = payload.get("id")
    sizes = payload.get("size")
    dimensions = payload.get("dimension", {})
    values = payload.get("value", {})

    if ids is None or sizes is None:
        raise ValueError("Eurostat payload missing id or size.")

    category_labels = {}

    for dim_id in ids:
        dim = dimensions.get(dim_id, {})
        category = dim.get("category", {})
        index = category.get("index", {})
        label = category.get("label", {})

        if not index:
            raise ValueError(f"Eurostat dimension {dim_id} missing category index.")

        inv = {pos: code for code, pos in index.items()}
        ordered_codes = [inv[i] for i in range(len(inv))]

        category_labels[dim_id] = [
            {
                "code": code,
                "label": label.get(code, code),
            }
            for code in ordered_codes
        ]

    rows = []
    total_size = int(np.prod(sizes))

    for flat_idx in range(total_size):
        if isinstance(values, dict):
            key = str(flat_idx)

            if key not in values:
                continue

            value = values[key]

        elif isinstance(values, list):
            if flat_idx >= len(values):
                continue

            value = values[flat_idx]

        else:
            continue

        if value is None:
            continue

        remainder = flat_idx
        coords = []

        for size in reversed(sizes):
            coords.append(remainder % size)
            remainder = remainder // size

        coords = list(reversed(coords))

        row = {}

        for dim_id, coord in zip(ids, coords):
            row[dim_id] = category_labels[dim_id][coord]["code"]
            row[f"{dim_id}_label"] = category_labels[dim_id][coord]["label"]

        row[value_name] = value
        rows.append(row)

    return pd.DataFrame(rows)

In [14]:
# ------------------------------------------------------
# EUROSTAT PUBLIC DEBT DOWNLOADER
# ------------------------------------------------------
# General government gross debt, % of GDP.
# Dataset: gov_10dd_edpt1
# Filters:
# freq=A
# unit=PC_GDP
# sector=S13
# na_item=GD
#
# No project-specific country filtering is applied here.

EUROSTAT_PUBLIC_DEBT_FILE = RAW_FILES_DIR / "Eurostat_Public_Debt_GDP_2002_2025.csv"


def download_eurostat_public_debt():
    dataset_label = "Eurostat_Public_Debt_GDP"
    file_name = EUROSTAT_PUBLIC_DEBT_FILE.name

    print("=" * 80)
    print(f"Processing Eurostat dataset: {file_name}")

    url = f"{EUROSTAT_BASE_URL}/gov_10dd_edpt1"

    params = {
        "format": "JSON",
        "lang": "en",
        "freq": "A",
        "unit": "PC_GDP",
        "sector": "S13",
        "na_item": "GD",
        "sinceTimePeriod": str(START_YEAR),
        "untilTimePeriod": str(END_YEAR),
    }

    response = request_with_retry(
        url=url,
        params=params,
        headers={"User-Agent": "EconomicResilienceProject/1.0 (Python/Requests)"},
    )

    if response.status_code != 200:
        raise RuntimeError(f"HTTP {response.status_code}: {response.text[:500]}")

    payload = response.json()

    df_long = parse_eurostat_jsonstat(payload, value_name="Value")

    df_long.to_csv(
        DIAGNOSTICS_DIR / "eurostat_public_debt_rich_output.csv",
        index=False,
    )

    if df_long.empty:
        raise ValueError("Eurostat public debt returned no rows.")

    if "geo" not in df_long.columns or "time" not in df_long.columns:
        raise ValueError(f"Eurostat public debt missing geo/time columns. Columns: {df_long.columns.tolist()}")

    df_tidy = pd.DataFrame({
        "Country": df_long["geo"].astype(str).str.strip().str.upper(),
        "Year": pd.to_numeric(df_long["time"], errors="coerce"),
        "Value": pd.to_numeric(df_long["Value"], errors="coerce"),
    })

    df_tidy = df_tidy.dropna(subset=["Year", "Value"]).copy()
    df_tidy["Year"] = df_tidy["Year"].astype(int)

    df_tidy = validate_tidy_country_year_value(
        df_tidy,
        dataset_name=dataset_label,
        stop_on_duplicates=True,
    )

    write_tidy_csv(df_tidy, EUROSTAT_PUBLIC_DEBT_FILE, overwrite=OVERWRITE_RAW_FILES)

    print(f"  -> Success: {len(df_tidy)} rows saved to {EUROSTAT_PUBLIC_DEBT_FILE}")

    add_acquisition_log(
        dataset_label=dataset_label,
        file_name=file_name,
        status="success",
        source_route="EUROSTAT_API",
        rows_raw=len(df_long),
        rows_saved=len(df_tidy),
        countries=df_tidy["Country"].nunique(),
        min_year=df_tidy["Year"].min(),
        max_year=df_tidy["Year"].max(),
        output_file=EUROSTAT_PUBLIC_DEBT_FILE,
        note="General government gross debt, % of GDP. Raw Eurostat geography codes preserved for Step 00 standardization.",
        url=url,
    )

    return df_tidy


try:
    eurostat_public_debt = download_eurostat_public_debt()

except Exception as e:
    print("\n" + "!" * 80)
    print("Eurostat public debt failed.")
    print("Error:", e)
    print("!" * 80 + "\n")

    add_failed_download(
        dataset_label="Eurostat_Public_Debt_GDP",
        source_route="EUROSTAT_API",
        error=e,
    )

    if STOP_ON_REQUIRED_FAILURE:
        raise

Processing Eurostat dataset: Eurostat_Public_Debt_GDP_2002_2025.csv
  -> Success: 744 rows saved to D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Raw\Raw_Files\Eurostat_Public_Debt_GDP_2002_2025.csv


In [15]:
# ------------------------------------------------------
# WORLDWIDE GOVERNANCE INDICATORS RAW DOWNLOAD
# ------------------------------------------------------
# Purpose:
# Download WGI raw bulk data into Data/Raw/Raw_Files.
#
# Important:
# This acquisition notebook only downloads and extracts the raw WGI package.
#
# WGI compilation, six-dimension extraction, coverage checks, and derived
# governance composite construction will happen later in:
#
# 03_WGI_Governance_Compilation.ipynb

WGI_ZIP_URL = "https://databank.worldbank.org/data/download/WGI_CSV.zip"

WGI_ZIP_FILE = RAW_FILES_DIR / "WGI_CSV.zip"
WGI_RAW_DIR = RAW_FILES_DIR / "WGI_2025_Raw"
WGI_RAW_DIR.mkdir(parents=True, exist_ok=True)


def download_wgi_raw_zip():
    dataset_label = "WGI_Governance_Raw"

    print("=" * 80)
    print("Downloading Worldwide Governance Indicators raw zip")
    print("URL:", WGI_ZIP_URL)

    response = request_with_retry(
        url=WGI_ZIP_URL,
        params=None,
        headers={"User-Agent": "EconomicResilienceProject/1.0 (Python/Requests)"},
    )

    if response.status_code != 200:
        raise RuntimeError(f"WGI download failed. HTTP {response.status_code}: {response.text[:500]}")

    WGI_ZIP_FILE.write_bytes(response.content)

    if not zipfile.is_zipfile(WGI_ZIP_FILE):
        preview_path = DIAGNOSTICS_DIR / "wgi_download_not_zip_preview.txt"
        preview_path.write_text(response.text[:5000], encoding="utf-8", errors="ignore")
        raise ValueError(f"WGI download did not produce a valid zip file. Preview saved to {preview_path}")

    with zipfile.ZipFile(WGI_ZIP_FILE, "r") as z:
        extracted_files = z.namelist()
        z.extractall(WGI_RAW_DIR)

    extracted_inventory = pd.DataFrame({
        "extracted_file": extracted_files
    })

    extracted_inventory.to_csv(
        DIAGNOSTICS_DIR / "wgi_extracted_file_inventory.csv",
        index=False,
    )

    wgi_package_inventory = pd.DataFrame([{
        "dataset_label": dataset_label,
        "file_name": WGI_ZIP_FILE.name,
        "exists": WGI_ZIP_FILE.exists(),
        "zip_size_bytes": WGI_ZIP_FILE.stat().st_size if WGI_ZIP_FILE.exists() else np.nan,
        "raw_folder": str(WGI_RAW_DIR),
        "extracted_file_count": len(extracted_files),
        "note": "Raw WGI package downloaded and extracted. Not parsed in 00A.",
    }])

    wgi_package_inventory.to_csv(
        DIAGNOSTICS_DIR / "wgi_raw_package_inventory.csv",
        index=False,
    )

    add_acquisition_log(
        dataset_label=dataset_label,
        file_name=WGI_ZIP_FILE.name,
        status="success",
        source_route="WORLD_BANK_DATABANK_BULK_DOWNLOAD",
        rows_raw=None,
        rows_saved=None,
        countries=None,
        min_year=None,
        max_year=None,
        output_file=WGI_ZIP_FILE,
        note="Raw WGI bulk CSV zip downloaded and extracted. Processing deferred to 03_WGI_Governance_Compilation.ipynb.",
        url=WGI_ZIP_URL,
    )

    print("WGI zip saved to:")
    print(WGI_ZIP_FILE.resolve())

    print("\nWGI files extracted to:")
    print(WGI_RAW_DIR.resolve())

    print("\nExtracted files:")
    for file in extracted_files:
        print("-", file)

    display(wgi_package_inventory)
    display(extracted_inventory)

    return extracted_inventory


try:
    wgi_extracted_inventory = download_wgi_raw_zip()

except Exception as e:
    print("\n" + "!" * 80)
    print("WGI raw download failed.")
    print("Error:", e)
    print("!" * 80 + "\n")

    add_failed_download(
        dataset_label="WGI_Governance_Raw",
        source_route="WORLD_BANK_DATABANK_BULK_DOWNLOAD",
        error=e,
        url=WGI_ZIP_URL,
    )

    if STOP_ON_REQUIRED_FAILURE:
        raise

URL: https://databank.worldbank.org/data/download/WGI_CSV.zip
WGI zip saved to:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Raw\Raw_Files\WGI_CSV.zip

WGI files extracted to:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Raw\Raw_Files\WGI_2025_Raw

Extracted files:
- WGICSV.csv
- WGICountry.csv
- WGISeries.csv


,dataset_label,file_name,exists,zip_size_bytes,raw_folder,extracted_file_count,note
0,WGI_Governance_Raw,WGI_CSV.zip,True,3661035,D:\Milano Bicocca\Course Materials\1st Year\02...,3,Raw WGI package downloaded and extracted. Not ...


,extracted_file
0,WGICSV.csv
1,WGICountry.csv
2,WGISeries.csv


In [16]:
# ------------------------------------------------------
# SAVE METHODOLOGICAL NOTES
# ------------------------------------------------------

methodological_notes = pd.DataFrame([
    {
        "topic": "Acquisition sample rule",
        "note": "No country/sample filtering is done in 00A. All available countries/geographies are pulled for each selected indicator definition. Country inclusion is decided later.",
    },
    {
        "topic": "Productivity correction",
        "note": "Previous productivity extraction matched PRICE_BASE = V current prices. The corrected extraction uses GDP per hour worked with PRICE_BASE = Q constant prices through the DBnomics OECD mirror.",
    },
    {
        "topic": "GDP growth source correction",
        "note": "OECD REST query for GDP growth returned NoResultsFound. GDP growth is therefore acquired from the DBnomics OECD mirror of DSD_NAAG@DF_NAAG_I using real GDP growth rate series pattern A.*.B1GQ_R_GR.PC.NAAG_I.",
    },
    {
        "topic": "Tertiary education correction",
        "note": "Tertiary education is rebuilt as adult tertiary attainment, age 25-64, not as current tertiary enrolment. This better represents structural human-capital stock.",
    },
    {
        "topic": "Energy dependency",
        "note": "World Bank energy import dependency is kept raw. Negative values are meaningful and should not be clipped.",
    },
    {
        "topic": "Public debt",
        "note": "Eurostat and World Bank public debt are both acquired as raw files. Canonical public debt should be built after country-code standardization, not inside raw acquisition.",
    },
    {
        "topic": "WGI governance",
        "note": "WGI is downloaded as a raw package only. Six-dimension extraction and any derived governance composite are deferred to the WGI compilation notebook.",
    },
])

methodological_notes.to_csv(
    DIAGNOSTICS_DIR / "methodological_notes_for_decision_log.csv",
    index=False,
)

display(methodological_notes)

,topic,note
0,Acquisition sample rule,No country/sample filtering is done in 00A. Al...
1,Productivity correction,Previous productivity extraction matched PRICE...
2,GDP growth source correction,OECD REST query for GDP growth returned NoResu...
3,Tertiary education correction,Tertiary education is rebuilt as adult tertiar...
4,Energy dependency,World Bank energy import dependency is kept ra...
5,Public debt,Eurostat and World Bank public debt are both a...
6,WGI governance,WGI is downloaded as a raw package only. Six-d...


In [17]:
# ------------------------------------------------------
# SAVE ACQUISITION DIAGNOSTICS
# ------------------------------------------------------

acquisition_log = pd.DataFrame(acquisition_log_rows)

if acquisition_log.empty:
    acquisition_log = pd.DataFrame(columns=[
        "run_id", "timestamp", "dataset_label", "file_name", "status",
        "source_route", "rows_raw", "rows_saved", "countries", "min_year",
        "max_year", "output_file", "note", "url"
    ])

failed_downloads = pd.DataFrame(failed_download_rows)

if failed_downloads.empty:
    failed_downloads = pd.DataFrame(columns=[
        "run_id", "timestamp", "dataset_label", "source_route", "error", "url"
    ])

acquisition_log.to_csv(
    DIAGNOSTICS_DIR / "acquisition_log.csv",
    index=False,
)

failed_downloads.to_csv(
    DIAGNOSTICS_DIR / "failed_downloads.csv",
    index=False,
)

display(acquisition_log)
display(failed_downloads)

,run_id,timestamp,dataset_label,file_name,status,source_route,rows_raw,rows_saved,countries,min_year,max_year,output_file,note,url
0,20260624_173824,2026-06-24 17:38:24,OECD_RD_GDP,OECD_RD_GDP_2002_2025.csv,success,OECD_REST,139178.0000,1044.0000,49.0000,2002.0000,2025.0000,D:\Milano Bicocca\Course Materials\1st Year\02...,R&D expenditure as percentage of GDP.,https://sdmx.oecd.org/public/rest/data/OECD.ST...
1,20260624_173824,2026-06-24 17:38:24,OECD_Inflation_CPI,OECD_Inflation_CPI_2002_2025.csv,success,OECD_REST,1257.0000,1257.0000,54.0000,2002.0000,2025.0000,D:\Milano Bicocca\Course Materials\1st Year\02...,"Consumer price inflation, annual growth rate.",https://sdmx.oecd.org/public/rest/data/OECD.SD...
2,20260624_173824,2026-06-24 17:38:24,OECD_Unemployment_Rate,OECD_Unemployment_Rate_2002_2025.csv,success,OECD_REST,332574.0000,1338.0000,59.0000,2002.0000,2025.0000,D:\Milano Bicocca\Course Materials\1st Year\02...,"Unemployment rate, total sex, total age, perce...",https://sdmx.oecd.org/public/rest/data/OECD.EL...
3,20260624_173824,2026-06-24 17:38:24,OECD_Tertiary_Education,OECD_Tertiary_Education_2002_2025.csv,success,OECD_REST,1044.0000,943.0000,51.0000,2002.0000,2024.0000,D:\Milano Bicocca\Course Materials\1st Year\02...,"Adult tertiary attainment: age 25-64, total se...",https://sdmx.oecd.org/public/rest/data/OECD.ED...
4,20260624_173824,2026-06-24 17:38:24,OECD_Productivity_GDP_per_Hour,OECD_Productivity_GDP_per_Hour_2002_2025.csv,success,DBNOMICS_OECD_MIRROR,991.0000,991.0000,44.0000,2002.0000,2024.0000,D:\Milano Bicocca\Course Materials\1st Year\02...,"GDP per hour worked, total activities, USD PPP...",DBnomics OECD/DSD_PDB@DF_PDB_LV
5,20260624_173824,2026-06-24 17:38:24,OECD_GDP_Growth,OECD_GDP_Growth_2002_2025.csv,success,DBNOMICS_OECD_MIRROR,1115.0000,1115.0000,47.0000,2002.0000,2025.0000,D:\Milano Bicocca\Course Materials\1st Year\02...,"Real gross domestic product growth rate, perce...",DBnomics OECD/DSD_NAAG@DF_NAAG_I
6,20260624_173824,2026-06-24 17:38:24,WorldBank_Public_Debt_GDP,WorldBank_Public_Debt_GDP_2002_2025.csv,success,WORLD_BANK_API,1160.0000,1160.0000,91.0000,2002.0000,2024.0000,D:\Milano Bicocca\Course Materials\1st Year\02...,"Central government debt, total (% of GDP).",https://api.worldbank.org/v2/country/all/indic...
7,20260624_173824,2026-06-24 17:38:24,WorldBank_Energy_Import_Dependency_Raw,WorldBank_Energy_Import_Dependency_Raw_2002_20...,success,WORLD_BANK_API,3798.0000,3798.0000,188.0000,2002.0000,2023.0000,D:\Milano Bicocca\Course Materials\1st Year\02...,"Energy imports, net (% of energy use). Raw val...",https://api.worldbank.org/v2/country/all/indic...
8,20260624_173824,2026-06-24 17:38:24,Eurostat_Public_Debt_GDP,Eurostat_Public_Debt_GDP_2002_2025.csv,success,EUROSTAT_API,744.0000,744.0000,31.0000,2002.0000,2025.0000,D:\Milano Bicocca\Course Materials\1st Year\02...,"General government gross debt, % of GDP. Raw E...",https://ec.europa.eu/eurostat/api/disseminatio...
9,20260624_173824,2026-06-24 17:38:24,WGI_Governance_Raw,WGI_CSV.zip,success,WORLD_BANK_DATABANK_BULK_DOWNLOAD,NaN,NaN,NaN,NaN,NaN,D:\Milano Bicocca\Course Materials\1st Year\02...,Raw WGI bulk CSV zip downloaded and extracted....,https://databank.worldbank.org/data/download/W...


,run_id,timestamp,dataset_label,source_route,error,url


In [18]:
# ------------------------------------------------------
# RAW FILE INVENTORY AFTER ACQUISITION
# ------------------------------------------------------

EXPECTED_CSV_RAW_FILES = [
    "OECD_RD_GDP_2002_2025.csv",
    "OECD_Inflation_CPI_2002_2025.csv",
    "OECD_Unemployment_Rate_2002_2025.csv",
    "OECD_Tertiary_Education_2002_2025.csv",
    "OECD_Productivity_GDP_per_Hour_2002_2025.csv",
    "OECD_GDP_Growth_2002_2025.csv",
    "Eurostat_Public_Debt_GDP_2002_2025.csv",
    "WorldBank_Public_Debt_GDP_2002_2025.csv",
    "WorldBank_Energy_Import_Dependency_Raw_2002_2025.csv",
]

inventory_rows = []

for file_name in EXPECTED_CSV_RAW_FILES:
    path = RAW_FILES_DIR / file_name
    summary = summarize_saved_csv(path, dataset_label=file_name.replace(".csv", ""))
    inventory_rows.append(summary)

raw_file_inventory = pd.DataFrame(inventory_rows)

raw_file_inventory.to_csv(
    DIAGNOSTICS_DIR / "raw_file_inventory_after_acquisition.csv",
    index=False,
)

wgi_package_inventory = pd.DataFrame([{
    "dataset_label": "WGI_Governance_Raw",
    "file_name": "WGI_CSV.zip",
    "exists": WGI_ZIP_FILE.exists(),
    "zip_size_bytes": WGI_ZIP_FILE.stat().st_size if WGI_ZIP_FILE.exists() else np.nan,
    "raw_folder": str(WGI_RAW_DIR),
    "note": "Raw WGI package downloaded and extracted if WGI acquisition succeeded. Not parsed in 00A.",
}])

wgi_package_inventory.to_csv(
    DIAGNOSTICS_DIR / "wgi_raw_package_inventory.csv",
    index=False,
)

display(raw_file_inventory)
display(wgi_package_inventory)

,dataset_label,file_name,exists,rows,columns,countries,min_year,max_year
0,OECD_RD_GDP_2002_2025,OECD_RD_GDP_2002_2025.csv,True,1044,3,49,2002,2025
1,OECD_Inflation_CPI_2002_2025,OECD_Inflation_CPI_2002_2025.csv,True,1257,3,54,2002,2025
2,OECD_Unemployment_Rate_2002_2025,OECD_Unemployment_Rate_2002_2025.csv,True,1338,3,59,2002,2025
3,OECD_Tertiary_Education_2002_2025,OECD_Tertiary_Education_2002_2025.csv,True,943,3,51,2002,2024
4,OECD_Productivity_GDP_per_Hour_2002_2025,OECD_Productivity_GDP_per_Hour_2002_2025.csv,True,991,3,44,2002,2024
5,OECD_GDP_Growth_2002_2025,OECD_GDP_Growth_2002_2025.csv,True,1115,3,47,2002,2025
6,Eurostat_Public_Debt_GDP_2002_2025,Eurostat_Public_Debt_GDP_2002_2025.csv,True,744,3,31,2002,2025
7,WorldBank_Public_Debt_GDP_2002_2025,WorldBank_Public_Debt_GDP_2002_2025.csv,True,1160,3,91,2002,2024
8,WorldBank_Energy_Import_Dependency_Raw_2002_2025,WorldBank_Energy_Import_Dependency_Raw_2002_20...,True,3798,3,188,2002,2023


,dataset_label,file_name,exists,zip_size_bytes,raw_folder,note
0,WGI_Governance_Raw,WGI_CSV.zip,True,3661035,D:\Milano Bicocca\Course Materials\1st Year\02...,Raw WGI package downloaded and extracted if WG...


In [19]:
# ------------------------------------------------------
# RAW FILE INVENTORY AFTER ACQUISITION
# ------------------------------------------------------

EXPECTED_CSV_RAW_FILES = [
    "OECD_RD_GDP_2002_2025.csv",
    "OECD_Inflation_CPI_2002_2025.csv",
    "OECD_Unemployment_Rate_2002_2025.csv",
    "OECD_Tertiary_Education_2002_2025.csv",
    "OECD_Productivity_GDP_per_Hour_2002_2025.csv",
    "OECD_GDP_Growth_2002_2025.csv",
    "Eurostat_Public_Debt_GDP_2002_2025.csv",
    "WorldBank_Public_Debt_GDP_2002_2025.csv",
    "WorldBank_Energy_Import_Dependency_Raw_2002_2025.csv",
]

inventory_rows = []

for file_name in EXPECTED_CSV_RAW_FILES:
    path = RAW_FILES_DIR / file_name
    summary = summarize_saved_csv(path, dataset_label=file_name.replace(".csv", ""))
    inventory_rows.append(summary)

raw_file_inventory = pd.DataFrame(inventory_rows)

raw_file_inventory.to_csv(
    DIAGNOSTICS_DIR / "raw_file_inventory_after_acquisition.csv",
    index=False,
)

wgi_package_inventory = pd.DataFrame([{
    "dataset_label": "WGI_Governance_Raw",
    "file_name": "WGI_CSV.zip",
    "exists": WGI_ZIP_FILE.exists(),
    "zip_size_bytes": WGI_ZIP_FILE.stat().st_size if WGI_ZIP_FILE.exists() else np.nan,
    "raw_folder": str(WGI_RAW_DIR),
    "note": "Raw WGI package downloaded and extracted if WGI acquisition succeeded. Not parsed in 00A.",
}])

wgi_package_inventory.to_csv(
    DIAGNOSTICS_DIR / "wgi_raw_package_inventory.csv",
    index=False,
)

display(raw_file_inventory)
display(wgi_package_inventory)

,dataset_label,file_name,exists,rows,columns,countries,min_year,max_year
0,OECD_RD_GDP_2002_2025,OECD_RD_GDP_2002_2025.csv,True,1044,3,49,2002,2025
1,OECD_Inflation_CPI_2002_2025,OECD_Inflation_CPI_2002_2025.csv,True,1257,3,54,2002,2025
2,OECD_Unemployment_Rate_2002_2025,OECD_Unemployment_Rate_2002_2025.csv,True,1338,3,59,2002,2025
3,OECD_Tertiary_Education_2002_2025,OECD_Tertiary_Education_2002_2025.csv,True,943,3,51,2002,2024
4,OECD_Productivity_GDP_per_Hour_2002_2025,OECD_Productivity_GDP_per_Hour_2002_2025.csv,True,991,3,44,2002,2024
5,OECD_GDP_Growth_2002_2025,OECD_GDP_Growth_2002_2025.csv,True,1115,3,47,2002,2025
6,Eurostat_Public_Debt_GDP_2002_2025,Eurostat_Public_Debt_GDP_2002_2025.csv,True,744,3,31,2002,2025
7,WorldBank_Public_Debt_GDP_2002_2025,WorldBank_Public_Debt_GDP_2002_2025.csv,True,1160,3,91,2002,2024
8,WorldBank_Energy_Import_Dependency_Raw_2002_2025,WorldBank_Energy_Import_Dependency_Raw_2002_20...,True,3798,3,188,2002,2023


,dataset_label,file_name,exists,zip_size_bytes,raw_folder,note
0,WGI_Governance_Raw,WGI_CSV.zip,True,3661035,D:\Milano Bicocca\Course Materials\1st Year\02...,Raw WGI package downloaded and extracted if WG...


In [20]:
# ------------------------------------------------------
# CREATE ACQUISITION AUDIT WORKBOOK
# ------------------------------------------------------

ACQUISITION_AUDIT_XLSX = AUDIT_DIR / "00A_Data_Acquisition_Audit.xlsx"

audit_sources = [
    {
        "sheet_name": "01_acquisition_log",
        "description": "Main acquisition log for all downloaded files.",
        "path": DIAGNOSTICS_DIR / "acquisition_log.csv",
    },
    {
        "sheet_name": "02_failed_downloads",
        "description": "Any failed downloads or source errors.",
        "path": DIAGNOSTICS_DIR / "failed_downloads.csv",
    },
    {
        "sheet_name": "03_raw_inventory",
        "description": "Inventory of expected CSV raw files after acquisition.",
        "path": DIAGNOSTICS_DIR / "raw_file_inventory_after_acquisition.csv",
    },
    {
        "sheet_name": "04_wgi_package",
        "description": "WGI raw package inventory.",
        "path": DIAGNOSTICS_DIR / "wgi_raw_package_inventory.csv",
    },
    {
        "sheet_name": "05_value_ranges",
        "description": "Basic range checks for every saved CSV raw file.",
        "path": DIAGNOSTICS_DIR / "value_range_sanity_checks.csv",
    },
    {
        "sheet_name": "06_method_notes",
        "description": "Methodological notes that should later be transferred to the project decision log.",
        "path": DIAGNOSTICS_DIR / "methodological_notes_for_decision_log.csv",
    },
    {
        "sheet_name": "07_productivity_discovery",
        "description": "DBnomics productivity series discovered for the selected constant-price definition.",
        "path": DIAGNOSTICS_DIR / "dbnomics_productivity_discovered_series_codes.csv",
    },
    {
        "sheet_name": "08_productivity_status",
        "description": "Country/geography-level status for productivity pulled through DBnomics OECD mirror.",
        "path": DIAGNOSTICS_DIR / "dbnomics_productivity_country_status.csv",
    },
    {
        "sheet_name": "09_productivity_sanity",
        "description": "Sanity check for final constant-price productivity file.",
        "path": DIAGNOSTICS_DIR / "dbnomics_productivity_sanity_check.csv",
    },
    {
        "sheet_name": "10_gdp_growth_discovery",
        "description": "DBnomics GDP growth series discovered for real GDP growth rate.",
        "path": DIAGNOSTICS_DIR / "dbnomics_gdp_growth_discovered_series_codes.csv",
    },
    {
        "sheet_name": "11_gdp_growth_status",
        "description": "Country/geography-level status for GDP growth pulled through DBnomics OECD mirror.",
        "path": DIAGNOSTICS_DIR / "dbnomics_gdp_growth_country_status.csv",
    },
    {
        "sheet_name": "12_gdp_growth_sanity",
        "description": "Sanity check for final GDP growth file.",
        "path": DIAGNOSTICS_DIR / "dbnomics_gdp_growth_sanity_check.csv",
    },
    {
        "sheet_name": "13_wgi_extracted",
        "description": "Files extracted from the WGI raw zip package.",
        "path": DIAGNOSTICS_DIR / "wgi_extracted_file_inventory.csv",
    },
]


def safe_sheet_name(name, used):
    cleaned = re.sub(r"[/\\*\[\]:?]", "_", str(name))[:31]
    base = cleaned
    i = 1

    while cleaned in used:
        suffix = f"_{i}"
        cleaned = base[:31 - len(suffix)] + suffix
        i += 1

    used.add(cleaned)
    return cleaned


def estimate_width(series, col_name, min_width=10, max_width=60):
    values = [str(col_name)] + series.dropna().astype(str).head(200).tolist()

    if not values:
        return min_width

    return max(min_width, min(max(len(v) for v in values) + 2, max_width))


used_sheets = set()

with pd.ExcelWriter(ACQUISITION_AUDIT_XLSX, engine="xlsxwriter") as writer:
    workbook = writer.book

    title_fmt = workbook.add_format({
        "bold": True,
        "font_size": 15,
        "font_color": "white",
        "bg_color": "#1F4E78",
        "align": "left",
        "valign": "vcenter",
    })

    desc_fmt = workbook.add_format({
        "text_wrap": True,
        "font_size": 10,
        "font_color": "#333333",
        "valign": "top",
    })

    header_fmt = workbook.add_format({
        "bold": True,
        "font_color": "white",
        "bg_color": "#5B9BD5",
        "border": 1,
        "align": "center",
        "valign": "vcenter",
        "text_wrap": True,
    })

    index_rows = []

    for item in audit_sources:
        path = Path(item["path"])

        if path.exists():
            try:
                df_temp = pd.read_csv(path)
                rows = len(df_temp)
                cols = len(df_temp.columns)
            except Exception:
                rows = np.nan
                cols = np.nan
        else:
            rows = 0
            cols = 0

        index_rows.append({
            "Sheet": item["sheet_name"],
            "Rows": rows,
            "Columns": cols,
            "Description": item["description"],
            "Path": str(path),
        })

    index_df = pd.DataFrame(index_rows)
    index_df.to_excel(writer, sheet_name="00_INDEX", index=False, startrow=5)

    ws = writer.sheets["00_INDEX"]
    ws.merge_range(0, 0, 0, max(4, len(index_df.columns) - 1), "00A Data Acquisition Audit", title_fmt)
    ws.merge_range(1, 0, 3, max(4, len(index_df.columns) - 1), "Audit workbook for raw data acquisition before standardization.", desc_fmt)

    for col_idx, col_name in enumerate(index_df.columns):
        ws.write(5, col_idx, col_name, header_fmt)
        ws.set_column(col_idx, col_idx, estimate_width(index_df[col_name], col_name))

    ws.autofilter(5, 0, 5 + len(index_df), len(index_df.columns) - 1)
    ws.freeze_panes(6, 0)

    for item in audit_sources:
        path = Path(item["path"])

        if path.exists():
            try:
                df_sheet = pd.read_csv(path)
            except Exception as e:
                df_sheet = pd.DataFrame({"message": [f"Could not read file: {e}"]})
        else:
            df_sheet = pd.DataFrame({"message": ["File not found."]})

        if df_sheet.empty:
            df_sheet = pd.DataFrame({"message": ["No rows in this table."]})

        sheet_name = safe_sheet_name(item["sheet_name"], used_sheets)

        df_sheet.to_excel(writer, sheet_name=sheet_name, index=False, startrow=4)

        ws = writer.sheets[sheet_name]
        max_col = max(4, len(df_sheet.columns) - 1)

        ws.merge_range(0, 0, 0, max_col, item["sheet_name"], title_fmt)
        ws.merge_range(1, 0, 3, max_col, item["description"], desc_fmt)

        for col_idx, col_name in enumerate(df_sheet.columns):
            ws.write(4, col_idx, col_name, header_fmt)
            ws.set_column(col_idx, col_idx, estimate_width(df_sheet[col_name], col_name))

        ws.autofilter(4, 0, 4 + len(df_sheet), len(df_sheet.columns) - 1)
        ws.freeze_panes(5, 0)

print("Acquisition audit workbook created:")
print(ACQUISITION_AUDIT_XLSX.resolve())

Acquisition audit workbook created:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Audit\00A_Data_Acquisition_Audit.xlsx


In [21]:
# ------------------------------------------------------
# FINAL COMPLETION CHECK
# ------------------------------------------------------

print("00 DATA ACQUISITION COMPLETE")
print("=" * 80)

print("\nRaw files folder:")
print(RAW_FILES_DIR.resolve())

print("\nDiagnostics folder:")
print(DIAGNOSTICS_DIR.resolve())

print("\nAudit workbook:")
print(ACQUISITION_AUDIT_XLSX.resolve())

print("\nExpected CSV raw files:")
for file_name in EXPECTED_CSV_RAW_FILES:
    path = RAW_FILES_DIR / file_name
    status = "FOUND" if path.exists() else "MISSING"
    print(f"- {status}: {file_name}")

print("\nExpected WGI raw package:")
print("-", "FOUND" if WGI_ZIP_FILE.exists() else "MISSING", ":", WGI_ZIP_FILE.name)
print("- WGI extracted folder:", WGI_RAW_DIR)

print("\nImportant methodological notes:")
print("1. No country/sample filtering is done in this acquisition notebook.")
print("2. Tertiary education is adult tertiary attainment, age 25-64, not tertiary enrolment.")
print("3. Productivity uses DBnomics OECD mirror with PRICE_BASE = Q constant prices.")
print("4. GDP growth uses DBnomics OECD mirror because the OECD REST query returned NoResultsFound.")
print("5. Energy import dependency is raw and not clipped; negative values are meaningful.")
print("6. Public debt canonicalization is postponed until after country-code standardization.")
print("7. WGI is downloaded raw only; WGI processing is deferred to the WGI compilation notebook.")
print("8. No imputation, direction alignment, or POSet-ready transformations are done here.")

print("\nNext notebook:")
print("01_Make_Raw_Files_Comparable.ipynb")

00 DATA ACQUISITION COMPLETE

Raw files folder:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Raw\Raw_Files

Diagnostics folder:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Diagnostics\00A_Data_Acquisition

Audit workbook:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Audit\00A_Data_Acquisition_Audit.xlsx

Expected CSV raw files:
- FOUND: OECD_RD_GDP_2002_2025.csv
- FOUND: OECD_Inflation_CPI_2002_2025.csv
- FOUND: OECD_Unemployment_Rate_2002_2025.csv
- FOUND: OECD_Tertiary_Education_2002_2025.csv
- FOUND: OECD_Productivity_GDP_per_Hour_2002_2025.csv
- FOUND: OECD_GDP_Growth_2002_2025.csv
- FOUND: Eurostat_Public_Debt_GDP_2002_2025.csv
- FOUND: WorldBank_Public_Debt_GDP_2002_2025.csv
- FOUND: WorldBank_Energy_Import_Dependency_Raw_2002_2025.csv

Expect